# DCC119 Mission 1 — Large Baseline

목표: 구급 Training에서 만든 **10,000 train / 3,000 valid** subset으로 baseline 재실험.

구성:
`4초 신고자 음성 → Log-Mel → Small CNN → M/F`

전제: `dcc119_m1_large_subset.zip`을 `/content`에 업로드.


In [1]:
# 0. Colab GPU 확인
!nvidia-smi

import torch, shutil, psutil
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2), "GB")
print("RAM:", round(psutil.virtual_memory().total/1024**3, 2), "GB")
print("Free /content:", round(shutil.disk_usage("/content").free/1024**3, 2), "GB")


Thu Sep 17 11:54:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. 필요한 패키지
!pip -q install librosa soundfile tqdm scikit-learn


In [4]:
# 2. ZIP 해제
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/dcc119_m1_large_subset.zip")
ROOT = Path("/content/dcc119_m1_large_subset")

if not ZIP_PATH.exists():
    raise FileNotFoundError("dcc119_m1_large_subset.zip을 /content에 업로드하세요.")
if not ROOT.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall("/content")

print("ROOT:", ROOT)
print("Train WAV:", len(list((ROOT/"train").glob("*/*.wav"))))
print("Valid WAV:", len(list((ROOT/"valid").glob("*/*.wav"))))


ROOT: /content/dcc119_m1_large_subset
Train WAV: 10000
Valid WAV: 3000


In [5]:
# 3. Metadata
import pandas as pd
train_meta = pd.read_csv(ROOT/"metadata/train.csv")
valid_meta = pd.read_csv(ROOT/"metadata/valid.csv")

print("Train:", train_meta.shape)
print("Valid:", valid_meta.shape)
print("Train gender:", train_meta["gender"].value_counts().to_dict())
print("Valid gender:", valid_meta["gender"].value_counts().to_dict())
print("Train conversations:", train_meta["conversation_id"].nunique())
print("Valid conversations:", valid_meta["conversation_id"].nunique())
print("Conversation overlap:",
      len(set(train_meta["conversation_id"].astype(str)) &
          set(valid_meta["conversation_id"].astype(str))))


Train: (10000, 8)
Valid: (3000, 8)
Train gender: {'M': 5000, 'F': 5000}
Valid gender: {'M': 1500, 'F': 1500}
Train conversations: 9071
Valid conversations: 2344
Conversation overlap: 0


In [8]:
# 4. Log-Mel cache
import librosa
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path

CACHE = ROOT / "mel_cache"
TRAIN_CACHE = CACHE / "train"
VALID_CACHE = CACHE / "valid"

TRAIN_CACHE.mkdir(parents=True, exist_ok=True)
VALID_CACHE.mkdir(parents=True, exist_ok=True)

SR = 16000
N_MELS = 64
N_FFT = 1024
HOP = 256


def get_local_wav_path(row, split):
    """
    PC에서 만들어진 output_path를 사용하지 않고,
    Colab의 ROOT를 기준으로 실제 WAV 경로를 만든다.

    구조:
    /content/dcc119_m1_large_subset/
      train/
        F/
        M/
      valid/
        F/
        M/
    """
    sample_id = str(row["sample_id"])
    gender = str(row["gender"])

    wav_path = (
        ROOT
        / split
        / gender
        / f"{sample_id}.wav"
    )

    return wav_path


def make_mel(path):
    y, sr = librosa.load(
        str(path),
        sr=SR,
        mono=True
    )

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP,
        n_mels=N_MELS,
        power=2.0
    )

    mel_db = librosa.power_to_db(
        mel,
        ref=np.max
    )

    return mel_db.astype(np.float32)


def cache_split(df, split, out_dir):

    failures = []

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"Mel cache: {split}"
    ):

        sample_id = str(row["sample_id"])

        # ★ Windows 경로 대신 Colab 내부 경로 사용
        wav_path = get_local_wav_path(
            row,
            split
        )

        cache_path = (
            out_dir
            / f"{sample_id}.npy"
        )

        if cache_path.exists():
            continue

        if not wav_path.exists():
            failures.append({
                "sample_id": sample_id,
                "wav_path": str(wav_path),
                "error": "WAV not found"
            })
            continue

        try:

            mel = make_mel(wav_path)

            np.save(
                cache_path,
                mel
            )

        except Exception as e:

            failures.append({
                "sample_id": sample_id,
                "wav_path": str(wav_path),
                "error": repr(e)
            })

    print(
        f"\n[{split}]"
    )

    print(
        "WAV 확인:",
        len(df) - len(failures)
    )

    print(
        "Cache:",
        len(list(out_dir.glob("*.npy")))
    )

    print(
        "실패:",
        len(failures)
    )

    if failures:
        print(
            "\n첫 번째 실패:"
        )
        print(
            failures[0]
        )

    return failures


train_cache_fail = cache_split(
    train_meta,
    "train",
    TRAIN_CACHE
)

valid_cache_fail = cache_split(
    valid_meta,
    "valid",
    VALID_CACHE
)

Mel cache: train:   0%|          | 0/10000 [00:00<?, ?it/s]


[train]
WAV 확인: 10000
Cache: 10000
실패: 0


Mel cache: valid:   0%|          | 0/3000 [00:00<?, ?it/s]


[valid]
WAV 확인: 3000
Cache: 3000
실패: 0


In [9]:
# 5. Dataset
from torch.utils.data import Dataset, DataLoader

LABEL2ID = {
    "F": 0,
    "M": 1
}

ID2LABEL = {
    0: "F",
    1: "M"
}


class MelDataset(Dataset):

    def __init__(
        self,
        df,
        cache_dir
    ):

        self.df = (
            df
            .reset_index(drop=True)
            .copy()
        )

        self.cache_dir = Path(
            cache_dir
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        sample_id = str(
            row["sample_id"]
        )

        gender = str(
            row["gender"]
        )

        cache_path = (
            self.cache_dir
            / f"{sample_id}.npy"
        )

        if not cache_path.exists():
            raise FileNotFoundError(
                f"Cache 파일이 없습니다:\n{cache_path}"
            )

        x = np.load(
            cache_path
        ).astype(np.float32)

        # [mel, time]
        # → [channel, mel, time]
        x = torch.from_numpy(
            x
        ).unsqueeze(0)

        y = torch.tensor(
            LABEL2ID[gender],
            dtype=torch.long
        )

        return (
            x,
            y,
            sample_id
        )


train_ds = MelDataset(
    train_meta,
    TRAIN_CACHE
)

valid_ds = MelDataset(
    valid_meta,
    VALID_CACHE
)


train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=128,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


# 샘플 확인
x, y, sid = train_ds[0]

print(
    "Feature shape:",
    tuple(x.shape)
)

print(
    "Label:",
    ID2LABEL[int(y)]
)

print(
    "Sample ID:",
    sid
)

Feature shape: (1, 64, 251)
Label: M
Sample ID: 651e4f2166d8da7dcba972ad_utt0003_M


In [10]:
# 6. Small CNN
import torch.nn as nn

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128,2))
    def forward(self,x):
        return self.fc(self.pool(self.features(x)))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmallCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print("Device:", DEVICE)


Device: cuda


In [11]:
# 7. 학습
import time

def run_epoch(loader, train):
    model.train(train)
    total_loss = total_correct = total_n = 0
    for x,y,_ in loader:
        x,y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        if train: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits,y)
            if train:
                loss.backward(); optimizer.step()
        total_loss += loss.item()*len(y)
        total_correct += (logits.argmax(1)==y).sum().item()
        total_n += len(y)
    return total_loss/total_n, total_correct/total_n

EPOCHS = 5
best_acc, best_state, hist = -1, None, []

for ep in range(1,EPOCHS+1):
    t=time.time()
    tl,ta = run_epoch(train_loader, True)
    vl,va = run_epoch(valid_loader, False)
    hist.append([ep,tl,ta,vl,va,time.time()-t])
    print(f"Epoch {ep} | {time.time()-t:.2f}s | Train {ta:.4f} | Valid {va:.4f}")
    if va > best_acc:
        best_acc = va
        best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

BEST = ROOT/"best_m1_smallcnn_large.pt"
torch.save({"model_state_dict":best_state,"best_valid_acc":best_acc,"history":hist}, BEST)
print("Best valid:", best_acc)
print("Saved:", BEST)


Epoch 1 | 14.67s | Train 0.6645 | Valid 0.6533
Epoch 2 | 11.70s | Train 0.6928 | Valid 0.6863
Epoch 3 | 11.80s | Train 0.7136 | Valid 0.6267
Epoch 4 | 11.73s | Train 0.7236 | Valid 0.6903
Epoch 5 | 11.68s | Train 0.7269 | Valid 0.6687
Best valid: 0.6903333333333334
Saved: /content/dcc119_m1_large_subset/best_m1_smallcnn_large.pt


## 8. ★ 성능 검증

개별 발화 Accuracy와 통화 단위 Majority Vote / Mean Probability를 비교한다.

`sample_id`에서 바로 conversation을 추정하지 않고 **metadata의 conversation_id를 merge**한다.


In [12]:
# 9. Best model 평가
ckpt = torch.load(BEST, map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()

rows=[]
with torch.no_grad():
    for x,y,sids in valid_loader:
        x=x.to(DEVICE, non_blocking=True)
        p=torch.softmax(model(x),dim=1).cpu().numpy()
        pred=p.argmax(1); yt=y.numpy()
        for sid, yi, pi, pp in zip(sids,yt,pred,p):
            rows.append({"sample_id":str(sid),"true_id":int(yi),"pred_id":int(pi),
                         "true_gender":ID2LABEL[int(yi)],"pred_gender":ID2LABEL[int(pi)],
                         "p_F":float(pp[0]),"p_M":float(pp[1])})
pred_df=pd.DataFrame(rows)
utt_acc=(pred_df.true_id==pred_df.pred_id).mean()
print(f"Utterance Accuracy: {utt_acc:.4f} ({utt_acc*100:.2f}%)")


Utterance Accuracy: 0.6903 (69.03%)


In [13]:
# 10. Conversation-level 검증
link = valid_meta[["sample_id","conversation_id","gender"]].copy()
link["sample_id"]=link["sample_id"].astype(str)
eval_df=pred_df.merge(link,on="sample_id",how="left",validate="one_to_one")

def majority(s):
    return s.value_counts().idxmax()

conv=[]
for cid,g in eval_df.groupby("conversation_id"):
    true=g.gender.iloc[0]
    maj=majority(g.pred_gender)
    pf,pm=g.p_F.mean(),g.p_M.mean()
    mp="F" if pf>=pm else "M"
    vc=g.pred_gender.value_counts()
    conv.append({
        "conversation_id":str(cid),"true_gender":true,
        "majority_pred":maj,"mean_prob_pred":mp,
        "n_utterances":len(g),"F_votes":int(vc.get("F",0)),
        "M_votes":int(vc.get("M",0)),
        "mean_p_F":pf,"mean_p_M":pm,
        "majority_correct":maj==true,"mean_prob_correct":mp==true})
conv_df=pd.DataFrame(conv)

maj_acc=conv_df.majority_correct.mean()
mean_acc=conv_df.mean_prob_correct.mean()
ties=int((conv_df.F_votes==conv_df.M_votes).sum())

print("Validation conversations:", len(conv_df))
print(f"Majority Vote Accuracy: {maj_acc:.4f} ({maj_acc*100:.2f}%)")
print(f"Mean Probability Accuracy: {mean_acc:.4f} ({mean_acc*100:.2f}%)")
print("Majority ties:", ties)
print("\nUtterances per conversation:")
print(conv_df.n_utterances.describe())


Validation conversations: 2344
Majority Vote Accuracy: 0.6911 (69.11%)
Mean Probability Accuracy: 0.6962 (69.62%)
Majority ties: 105

Utterances per conversation:
count    2344.000000
mean        1.279863
std         0.575651
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         6.000000
Name: n_utterances, dtype: float64


In [14]:
# 11. 최종 요약
print("===== Mission 1 LARGE baseline =====")
print("Train:", len(train_meta))
print("Valid utterances:", len(pred_df))
print("Valid conversations:", len(conv_df))
print(f"Utterance Accuracy: {utt_acc:.4f} ({utt_acc*100:.2f}%)")
print(f"Majority Conversation Accuracy: {maj_acc:.4f} ({maj_acc*100:.2f}%)")
print(f"Mean Probability Conversation Accuracy: {mean_acc:.4f} ({mean_acc*100:.2f}%)")
print("Train/Valid conversation overlap:",
      len(set(train_meta.conversation_id.astype(str)) &
          set(valid_meta.conversation_id.astype(str))))
print("Majority ties:", ties)


===== Mission 1 LARGE baseline =====
Train: 10000
Valid utterances: 3000
Valid conversations: 2344
Utterance Accuracy: 0.6903 (69.03%)
Majority Conversation Accuracy: 0.6911 (69.11%)
Mean Probability Conversation Accuracy: 0.6962 (69.62%)
Train/Valid conversation overlap: 0
Majority ties: 105


In [15]:
# 학습 결과 자세히 확인

if "history_df" in globals():
    display(history_df)

elif "hist" in globals():
    history_df = pd.DataFrame(
        hist,
        columns=[
            "epoch",
            "train_loss",
            "train_acc",
            "valid_loss",
            "valid_acc",
            "time_sec"
        ]
    )
    display(history_df)

else:
    print("학습 history 변수를 찾지 못했습니다.")

,epoch,train_loss,train_acc,valid_loss,valid_acc,time_sec
0,1,0.610865,0.6645,0.614509,0.653333,14.671253
1,2,0.583662,0.6928,0.624462,0.686333,11.695395
2,3,0.562404,0.7136,0.686190,0.626667,11.801980
3,4,0.552321,0.7236,0.574523,0.690333,11.732346
4,5,0.546052,0.7269,0.574044,0.668667,11.676108


In [17]:
# 7. 20 Epoch 재학습
model = SmallCNN().to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

EPOCHS = 20

best_valid_acc = -1
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):

    t = time.time()

    train_loss, train_acc = run_epoch(
        train_loader,
        True
    )

    valid_loss, valid_acc = run_epoch(
        valid_loader,
        False
    )

    elapsed = time.time() - t

    history.append([
        epoch,
        train_loss,
        train_acc,
        valid_loss,
        valid_acc,
        elapsed
    ])

    print(
        f"Epoch {epoch:02d} | "
        f"{elapsed:.2f}s | "
        f"Train Acc {train_acc:.4f} | "
        f"Valid Acc {valid_acc:.4f}"
    )

    if valid_acc > best_valid_acc:

        best_valid_acc = valid_acc

        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

history_df = pd.DataFrame(
    history,
    columns=[
        "epoch",
        "train_loss",
        "train_acc",
        "valid_loss",
        "valid_acc",
        "time_sec"
    ]
)

display(history_df)

print(
    f"Best Valid Accuracy: "
    f"{best_valid_acc:.4f} "
    f"({best_valid_acc*100:.2f}%)"
)

Epoch 01 | 11.78s | Train Acc 0.6641 | Valid Acc 0.6623
Epoch 02 | 11.69s | Train Acc 0.7049 | Valid Acc 0.5503
Epoch 03 | 11.67s | Train Acc 0.7128 | Valid Acc 0.6563
Epoch 04 | 11.72s | Train Acc 0.7214 | Valid Acc 0.7160
Epoch 05 | 11.71s | Train Acc 0.7274 | Valid Acc 0.7370
Epoch 06 | 11.75s | Train Acc 0.7256 | Valid Acc 0.5133
Epoch 07 | 12.17s | Train Acc 0.7351 | Valid Acc 0.5450
Epoch 08 | 11.80s | Train Acc 0.7361 | Valid Acc 0.5650
Epoch 09 | 11.88s | Train Acc 0.7455 | Valid Acc 0.7360
Epoch 10 | 11.98s | Train Acc 0.7502 | Valid Acc 0.7223
Epoch 11 | 12.11s | Train Acc 0.7560 | Valid Acc 0.6467
Epoch 12 | 12.06s | Train Acc 0.7527 | Valid Acc 0.7560
Epoch 13 | 12.09s | Train Acc 0.7572 | Valid Acc 0.7307
Epoch 14 | 12.11s | Train Acc 0.7681 | Valid Acc 0.6973
Epoch 15 | 11.91s | Train Acc 0.7661 | Valid Acc 0.5647
Epoch 16 | 11.96s | Train Acc 0.7700 | Valid Acc 0.7200
Epoch 17 | 11.82s | Train Acc 0.7740 | Valid Acc 0.7507
Epoch 18 | 11.81s | Train Acc 0.7785 | Valid Acc

,epoch,train_loss,train_acc,valid_loss,valid_acc,time_sec
0,1,0.612266,0.6641,0.608513,0.662333,11.777489
1,2,0.576496,0.7049,0.813564,0.550333,11.691671
2,3,0.560175,0.7128,0.629946,0.656333,11.667370
3,4,0.556661,0.7214,0.588559,0.716000,11.721373
4,5,0.545807,0.7274,0.530076,0.737000,11.714756
5,6,0.546090,0.7256,1.197298,0.513333,11.751614
6,7,0.533259,0.7351,0.976331,0.545000,12.173353
7,8,0.529187,0.7361,0.901397,0.565000,11.795620
8,9,0.522072,0.7455,0.529910,0.736000,11.880793
9,10,0.513668,0.7502,0.531471,0.722333,11.982452


Best Valid Accuracy: 0.7560 (75.60%)


In [18]:
# Best Epoch 모델로 복원
model.load_state_dict(best_state)
model = model.to(DEVICE)
model.eval()

print("Best model restored")
print(f"Best validation accuracy: {best_valid_acc:.4f}")

Best model restored
Best validation accuracy: 0.7560


In [19]:
# Best model 상세 평가
from sklearn.metrics import confusion_matrix, classification_report

model.eval()

rows = []

with torch.no_grad():
    for x, y, sample_ids in valid_loader:

        x = x.to(DEVICE, non_blocking=True)

        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()

        pred_ids = probs.argmax(axis=1)
        true_ids = y.numpy()

        for sid, true_id, pred_id, prob in zip(
            sample_ids,
            true_ids,
            pred_ids,
            probs
        ):
            rows.append({
                "sample_id": str(sid),
                "true_id": int(true_id),
                "pred_id": int(pred_id),
                "true_gender": ID2LABEL[int(true_id)],
                "pred_gender": ID2LABEL[int(pred_id)],
                "p_F": float(prob[0]),
                "p_M": float(prob[1]),
            })

pred_df = pd.DataFrame(rows)

# ---------------------------------------
# 1. Utterance Accuracy
# ---------------------------------------

utterance_acc = (
    pred_df["true_id"] == pred_df["pred_id"]
).mean()

print("=" * 50)
print("1. Utterance-level")
print("=" * 50)

print(
    f"Accuracy: {utterance_acc:.4f} "
    f"({utterance_acc*100:.2f}%)"
)

# ---------------------------------------
# 2. Confusion Matrix
# ---------------------------------------

cm = confusion_matrix(
    pred_df["true_id"],
    pred_df["pred_id"],
    labels=[0, 1]
)

print("\nConfusion Matrix")
print("          Pred F   Pred M")
print(f"True F    {cm[0,0]:7d} {cm[0,1]:8d}")
print(f"True M    {cm[1,0]:7d} {cm[1,1]:8d}")

# ---------------------------------------
# 3. M/F별 precision / recall / f1
# ---------------------------------------

print("\nClassification Report")

print(
    classification_report(
        pred_df["true_id"],
        pred_df["pred_id"],
        labels=[0, 1],
        target_names=["F", "M"],
        digits=4
    )
)

# ---------------------------------------
# 4. Conversation 정보 붙이기
# ---------------------------------------

link_df = valid_meta[
    ["sample_id", "conversation_id", "gender"]
].copy()

link_df["sample_id"] = (
    link_df["sample_id"].astype(str)
)

eval_df = pred_df.merge(
    link_df,
    on="sample_id",
    how="left",
    validate="one_to_one"
)

# ---------------------------------------
# 5. Conversation-level 평가
# ---------------------------------------

def majority_label(series):
    return series.value_counts().idxmax()

conversation_rows = []

for conversation_id, group in eval_df.groupby(
    "conversation_id"
):

    true_gender = group["gender"].iloc[0]

    # Majority Vote
    majority_pred = majority_label(
        group["pred_gender"]
    )

    # Mean Probability
    mean_p_F = group["p_F"].mean()
    mean_p_M = group["p_M"].mean()

    mean_prob_pred = (
        "F"
        if mean_p_F >= mean_p_M
        else "M"
    )

    vote_counts = (
        group["pred_gender"]
        .value_counts()
    )

    f_votes = int(
        vote_counts.get("F", 0)
    )

    m_votes = int(
        vote_counts.get("M", 0)
    )

    conversation_rows.append({
        "conversation_id":
            str(conversation_id),

        "true_gender":
            true_gender,

        "majority_pred":
            majority_pred,

        "mean_prob_pred":
            mean_prob_pred,

        "n_utterances":
            len(group),

        "F_votes":
            f_votes,

        "M_votes":
            m_votes,

        "mean_p_F":
            mean_p_F,

        "mean_p_M":
            mean_p_M,

        "majority_correct":
            majority_pred == true_gender,

        "mean_prob_correct":
            mean_prob_pred == true_gender,
    })

conv_df = pd.DataFrame(
    conversation_rows
)

majority_acc = (
    conv_df["majority_correct"]
    .mean()
)

mean_prob_acc = (
    conv_df["mean_prob_correct"]
    .mean()
)

ties = int(
    (
        conv_df["F_votes"]
        ==
        conv_df["M_votes"]
    ).sum()
)

print("\n" + "=" * 50)
print("2. Conversation-level")
print("=" * 50)

print(
    "Conversation count:",
    len(conv_df)
)

print(
    f"Majority Vote Accuracy: "
    f"{majority_acc:.4f} "
    f"({majority_acc*100:.2f}%)"
)

print(
    f"Mean Probability Accuracy: "
    f"{mean_prob_acc:.4f} "
    f"({mean_prob_acc*100:.2f}%)"
)

print(
    "Majority ties:",
    ties
)

print("\nUtterances per conversation")
print(
    conv_df["n_utterances"].describe()
)

1. Utterance-level
Accuracy: 0.7560 (75.60%)

Confusion Matrix
          Pred F   Pred M
True F       1317      183
True M        549      951

Classification Report
              precision    recall  f1-score   support

           F     0.7058    0.8780    0.7825      1500
           M     0.8386    0.6340    0.7221      1500

    accuracy                         0.7560      3000
   macro avg     0.7722    0.7560    0.7523      3000
weighted avg     0.7722    0.7560    0.7523      3000


2. Conversation-level
Conversation count: 2344
Majority Vote Accuracy: 0.7573 (75.73%)
Mean Probability Accuracy: 0.7654 (76.54%)
Majority ties: 99

Utterances per conversation
count    2344.000000
mean        1.279863
std         0.575651
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         6.000000
Name: n_utterances, dtype: float64
